# Hybrid CNN-LSTM Stock Price Predictor Model

In [1]:
import pandas as pd
import numpy as np
from keras.layers import Input, Conv1D, Dense, BatchNormalization, Dropout, LSTM
from keras.models import Model

## Data

In [3]:
df = pd.read_csv("data/all_data.csv")
df.drop(columns=["date"], inplace=True)

# Create numerical representation of labels for model to use
labels = sorted(list(set(df["label"])))
labels_indices = dict((l, i) for i, l in enumerate(labels))
indices_labels = dict((i, l) for i, l in enumerate(labels))

# Down=0, Flat=1, Up=2  (alphabetical order)
df["label_num"] = df["label"].map(labels_indices)

In [4]:
X, y = [], []
features = df.drop(columns=["label", "label_num"]).values
label_nums = df["label_num"].values
window_size = 20
num_features = len(df.drop(columns=["label", "label_num"]).columns)

# Convert data into 3D for CNN (samples, timesteps, features)
for i in range(0, len(df) - window_size): 
        X.append(features[i : i + window_size]) # Add a 2D section of data of all features from the days within the current window size
        y.append(label_nums[i + window_size]) # Add the labels for each day in the window size

X = np.array(X)
y = np.array(y)

X.shape # Shape should be ((len(df) - window_size), window_size, num_features)
y.shape # Shape should be ((len(df) - window_size),)


(1187,)

In [5]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## Model

All numbers are not real yet

In [6]:
inputs = Input(shape=(window_size,num_features))

# CNN Block
conv1 = Conv1D(filters=64, kernel_size=3, activation='relu')(inputs)
conv1 = BatchNormalization()(conv1)
conv1 = Dropout(0.2)(conv1)

conv2 = Conv1D(filters=64, kernel_size=5, activation='relu')(conv1)
conv2 = BatchNormalization()(conv2)
conv2 = Dropout(0.2)(conv2)

# LSTM Block
lstm = LSTM(5)(conv2)

# Dense layers and output
dense1 = Dense(5, activation='relu')(lstm)
outputs = Dense(3, activation='softmax')(dense1)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 66)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 18, 64)         │        12,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 18, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 14, 64)         │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5)              │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,240 (137.66 KB)

 Trainable params: 34,984 (136.66 KB)

 Non-trainable params: 256 (1.00 KB)

In [7]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=128, epochs=15, validation_data=(X_test, y_test))

score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Epoch 1/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.3446 - loss: 1.1093 - val_accuracy: 0.2941 - val_loss: 1.1351
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3319 - loss: 1.1058 - val_accuracy: 0.2941 - val_loss: 1.1213
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3846 - loss: 1.0882 - val_accuracy: 0.3025 - val_loss: 1.1160
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3836 - loss: 1.0858 - val_accuracy: 0.3235 - val_loss: 1.1044
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3836 - loss: 1.0807 - val_accuracy: 0.3235 - val_loss: 1.1004
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4004 - loss: 1.0758 - val_accuracy: 0.3571 - val_loss: 1.0967
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.4236 - loss: 1.0691 - val_accuracy: 0.3487 - val_loss: 1.0931
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4131 - loss: 1.0681 - val_accuracy: 0.3487 - val_loss: 1.0936


In [8]:
import numpy as np

sequence = np.random.random((100,))
print(sequence)

a = [1, 2, 3, 5, 6, 6, 7]
a[::2]

[0.40441862 0.43398191 0.57359056 0.77847026 0.05661276 0.75146542
 0.94402641 0.08802316 0.34882293 0.27085161 0.24114851 0.3552876
 0.65344326 0.7329425  0.6090724  0.94902296 0.97631672 0.29692182
 0.70400786 0.08255805 0.4201099  0.2317194  0.60793908 0.29047599
 0.35006679 0.027737   0.22684198 0.33741764 0.62126629 0.86790128
 0.74630561 0.05389228 0.23658086 0.87118133 0.30092836 0.15652642
 0.98870036 0.01655676 0.26721439 0.82655743 0.20455662 0.22131834
 0.95115697 0.53717757 0.15099057 0.61172822 0.94504777 0.24851111
 0.40882438 0.74776383 0.9721652  0.26615159 0.43924294 0.6893574
 0.86735811 0.54810616 0.86999636 0.33537412 0.9140147  0.6375015
 0.71589457 0.95914733 0.8271647  0.6082593  0.69666909 0.71423604
 0.14948182 0.20874758 0.70241574 0.2150722  0.51890544 0.62109145
 0.1762454  0.06541717 0.21867296 0.96203326 0.0267159  0.0951901
 0.66180515 0.50661414 0.52902866 0.56124534 0.93677029 0.22040961
 0.91205982 0.5912841  0.55226234 0.4118685  0.73405636 0.13117188

[1, 3, 6, 7]

## Training Model